In [1]:
import chess, chess.engine, os, stat
from policy import *
import random
from discrim import *

2026-04-20 19:56:25.533727: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-20 19:56:25.535453: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-20 19:56:25.575007: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-20 19:56:25.575850: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-20 19:56:26.808796: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

POLICY V8 (JOINT)


In [2]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Contempt': 0,
 'Min Split Depth': 0,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Minimum Thinking Time': 20,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Threads': 1,
 'Hash': 256}

In [3]:
games= load_json("./data/Bijay_1549_games.json")
print(len(games))

Loading games: 100%|██████████| 392/392 [00:00<00:00, 485.70it/s]

392


In [4]:
def simulate_games(agent, sf, num_games=400, file_dir="./data", file_postfix="0"):

    games_data = []

    for i in range(num_games):
        board = chess.Board()
        moves = []

        while not board.is_game_over():
            if board.turn == chess.WHITE:
                move = agent.act(board)
            else:
                sf.set_fen_position(board.fen())
                best = sf.get_best_move()

                if best is None:
                    break

                move = chess.Move.from_uci(best)

            board.push(move)
            moves.append(move.uci())

        game_data = {
            "event": "Agent vs Stockfish",
            "round": i + 1,
            "white": f"Mimic Agent of {agent.id}",
            "black": "Stockfish",
            "result": board.result(),
            "moves": moves
        }

        games_data.append(game_data)

    # 🔥 normalize postfix here (IMPORTANT)
    file_postfix = str(file_postfix).replace(".", "_")

    file_path = f"{file_dir}/{agent.id}_agent_vs_stockfish_{file_postfix}.json"

    with open(file_path, "w") as f:
        json.dump(games_data, f, indent=4)

    return file_path 

In [5]:
def overall_similarity_pipeline(json_A, json_B, player_A, player_B):

    print(f"\n--- FULL PIPELINE: {player_A} vs {player_B} ---\n")

    # ============================================================
    # 🔧 FIX: normalize JSON INSIDE PIPELINE (list → string)
    # ============================================================
    def normalize_json(path):
        with open(path, "r") as f:
            games = json.load(f)

        for g in games:
            if isinstance(g.get("moves"), list):
                g["moves"] = " ".join(g["moves"])

        return games

    # Write temporary cleaned versions (no external preprocessing step)
    import tempfile

    def write_temp(games):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".json")
        json.dump(games, tmp)
        tmp.close()
        return tmp.name

    clean_A = write_temp(normalize_json(json_A))
    clean_B = write_temp(normalize_json(json_B))

    # ============================================================
    # ORIGINAL PIPELINE (UNCHANGED LOGIC BELOW)
    # ============================================================
    b_A, m_A, l_A = load_json_game_sequences(clean_A, player_A, 1.0)
    b_B, m_B, l_B = load_json_game_sequences(clean_B, player_B, 0.0)

    min_games = min(len(l_A), len(l_B))
    if min_games == 0:
        print("Not enough usable games.")
        return None

    b_A, m_A, l_A = b_A[:min_games], m_A[:min_games], l_A[:min_games]
    b_B, m_B, l_B = b_B[:min_games], m_B[:min_games], l_B[:min_games]

    raw_boards = b_A + b_B
    raw_moves  = m_A + m_B
    raw_labels = l_A + l_B

    combined = list(zip(raw_boards, raw_moves, raw_labels))
    random.shuffle(combined)
    raw_boards, raw_moves, raw_labels = zip(*combined)

    all_boards = np.array(raw_boards)
    all_moves  = np.array(raw_moves)
    all_labels = np.array(raw_labels)

    all_moves_onehot = tf.one_hot(all_moves, NUM_MOVES)

    model = build_style_classifier()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        x={"board_seq": all_boards, "move_seq": all_moves_onehot},
        y=all_labels,
        batch_size=32,
        epochs=20,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    similarity = compute_overall_similarity(
        clean_A, clean_B, player_A, player_B, model
    )

    print(f"Overall playstyle similarity: {similarity:.2f}%")
    os.remove(clean_A)
    os.remove(clean_B)
    return similarity

In [6]:
def hyper_tuning(games,player_name, sf, file_dir="./data"):
    a_values = np.linspace(0, 1, 11)[::-1]
    
    best_a = None
    best_score = float("-inf")
    agent = Agent(player_name,stockfish_path=r"./stockfish/src/stockfish")

    for a in a_values:
        try:

            agent.train(games,alpha=a)
            
            player_file_path = f"{file_dir}/{agent.id}_games.json"
            agent_file_path = simulate_games(
                agent,
                sf,
                400,
                file_dir=file_dir,
                file_postfix=f"_a_{a:.2f}"
            )
            
            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent.id}",
                f"Mimic Agent of {agent.id}"
            )

            print(f"a={a:.2f}, score={score:.3f}")

            if score > best_score:
                best_score = score
                best_a = a
        
        finally:
            # 🔥 CRITICAL: prevent Colab crashes
            import gc
            tf.keras.backend.clear_session()
            gc.collect()

    print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")

In [ ]:
hyper_tuning(games,"Bijay_1549",sf)

/storage/home/jmy5612/model V8/policy.py:123: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_evaluation will still return full strength Stockfish's evaluation of the position.
  info = self.sf.get_evaluation()
/storage/home/jmy5612/model V8/policy.py:270: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_top_moves will still return the top moves of full strength Stockfish.
  top = self.sf.get_top_moves(top_k)


Epoch 1/20
783/783 [==============================] - 115s 143ms/step - loss: 0.0027 - accuracy: 0.0912
Epoch 2/20
783/783 [==============================] - 111s 142ms/step - loss: 0.0020 - accuracy: 0.0960
Epoch 3/20
783/783 [==============================] - 111s 142ms/step - loss: 0.0018 - accuracy: 0.0979
Epoch 4/20
783/783 [==============================] - 112s 143ms/step - loss: 0.0015 - accuracy: 0.0980
Epoch 5/20
783/783 [==============================] - 111s 142ms/step - loss: 0.0013 - accuracy: 0.0981
Epoch 6/20
783/783 [==============================] - 112s 143ms/step - loss: 0.0011 - accuracy: 0.0986
Epoch 7/20
783/783 [==============================] - 112s 142ms/step - loss: 8.2705e-04 - accuracy: 0.0991
Epoch 8/20
783/783 [==============================] - 111s 142ms/step - loss: 6.3044e-04 - accuracy: 0.0992
Epoch 9/20
783/783 [==============================] - 112s 143ms/step - loss: 4.6163e-04 - accuracy: 0.0995
Epoch 10/20
783/783 [==============================]

In [ ]:
def play_sf_vs_sf(sf, num_games=400, file_dir="./data"):
    all_games = []

    for i in range(num_games):
        print(f"Game {i+1}/{num_games}")
        board = chess.Board()
        moves = []
        move_number = 1

        while not board.is_game_over():
            fen = board.fen()
            sf.set_fen_position(fen)
            move = sf.get_best_move()
            if move is None:
                break
            moves.append((board.turn, move_number, board.san(chess.Move.from_uci(move))))
            board.push(chess.Move.from_uci(move))
            if board.turn == chess.WHITE:
                move_number += 1

        pgn_moves = []
        for turn, num, san in moves:
            if turn == chess.WHITE:
                pgn_moves.append(f"{num}. {san}")
            else:
                pgn_moves.append(san)

        result = board.result()
        if result == "*":
            result = "1/2-1/2"

        all_games.append({
            "event": "SF vs SF",
            "white": "sf_white",
            "black": "Stockfish",
            "result": result,
            "moves": " ".join(pgn_moves)
        })

        print(f"Result: {result}")

    file_path = f"{file_dir}/sf_white_games.json"
    os.makedirs(file_dir, exist_ok=True)
    with open(file_path, "w") as f:
        json.dump(all_games, f, indent=4)
    print(f"Saved {num_games} games to {file_path}")

    return file_path

In [ ]:
def get_sf_white_similarity(sf, player_name, file_dir="./data"):
    sf_json = play_sf_vs_sf(sf, file_dir=file_dir)
    player_json = f"{file_dir}/{player_name}_games.json"

    similarity = overall_similarity_pipeline(
        json_A=player_json,
        json_B=sf_json,
        player_A=player_name,
        player_B="sf_white"
    )

    print(f"Stockfish white vs {player_name} similarity: {similarity:.2f}%")
    return similarity

similarity = get_sf_white_similarity(
    sf,
    player_name="Bijay_1549"
)